# Audio AI with OpenAI APIs

This notebook runs a modular audio pipeline where every model call goes through the OpenAI API:

```text
Audio file
-> preprocessing
-> OpenAI speech-to-text
-> transcript embedding
-> retrieval over past calls
-> OpenAI LLM answer
-> OpenAI text-to-speech
-> spoken output
```

The key teaching point: **audio does not become an answer in one step.** The system first turns speech into text, turns text into embeddings for retrieval, uses an LLM to generate an answer from evidence, and turns the answer back into audio.

## Teaching goal

Students should leave with one clear mental model:

```text
speech -> transcript -> embedding -> retrieval -> grounded answer -> spoken response
```

They should also understand the boundary between the steps:

| Step | Purpose |
|---|---|
| Speech-to-text | Convert speech into a transcript |
| Embeddings | Make transcripts searchable |
| Retrieval | Find similar prior calls or context |
| LLM | Summarize, classify, and suggest action |
| Text-to-speech | Convert the final answer into audio |

## Setup

Install the Python packages before the session:

```bash
pip install --upgrade openai python-dotenv pandas numpy scikit-learn librosa soundfile ipython
```

Install FFmpeg before the session. It is still useful even when the AI models run through OpenAI: it handles audio conversion, resampling, and extracting audio from video.

```bash
# macOS
brew install ffmpeg

# Ubuntu / Debian
sudo apt-get update
sudo apt-get install ffmpeg
```

Create a `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=your_api_key_here
```

Optional model overrides:

```env
OPENAI_ANSWER_MODEL=gpt-5-mini
OPENAI_STT_MODEL=gpt-4o-mini-transcribe
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
OPENAI_TTS_MODEL=gpt-4o-mini-tts
OPENAI_TTS_VOICE=coral
```

Do not commit `.env` to Git and do not paste the API key into the notebook.

Bring one short WAV, M4A, or MP3 file named `sample_support_call.wav`. Recommended spoken content:

```text
Hi, I cannot log into my account. The password reset email never arrives, and I need access today.
```

Keep it under 15 seconds. A short, clean recording makes the demo more predictable and keeps API cost low.

Before class: set the notebook font to 18pt+, test speaker volume, and keep `sample_support_call.wav` and `.env` next to this notebook.

In [ ]:
# Optional install cell.
# Uncomment and run this cell only if the environment is missing packages.

# %pip install -q --upgrade openai python-dotenv pandas numpy scikit-learn librosa soundfile ipython

## Step 1 — Verify the setup

This cell checks the API key, FFmpeg, and the OpenAI client. It also prints which models your project can actually reach.

**Important for this demo:** speech-to-text and text-to-speech are separate model families in OpenAI. If your project has no access to them, the API returns `403 PermissionDeniedError`. The notebook then falls back to a local model so the class can continue.

In [ ]:
import os
import shutil

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

print("OpenAI key set:", bool(os.environ.get("OPENAI_API_KEY")))
print("FFmpeg available:", bool(shutil.which("ffmpeg")))

client = OpenAI()
print("OpenAI client ready")

try:
    available = sorted(m.id for m in client.models.list())
    print("Models available to this project:", available)
except Exception as exc:
    print("Could not list models:", repr(exc))

## Step 2 — Imports and constants

We define the input audio file, the cleaned audio file, the text-to-speech output file, and the four model names used in the pipeline.

> Say: "In the local version we used separate local models. In this version, the model calls go through OpenAI: one call for transcription, one for embeddings, one for answer generation, and one for speech generation."

In [ ]:
from pathlib import Path
import re
import subprocess
import time

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Audio, display

INPUT_AUDIO = Path("sample_support_call.wav")
CLEAN_AUDIO = Path("sample_16k_mono.wav")
TTS_OUTPUT = Path("answer_openai.wav")

STT_MODEL = os.getenv("OPENAI_STT_MODEL", "gpt-4o-mini-transcribe")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
ANSWER_MODEL = os.getenv("OPENAI_ANSWER_MODEL", "gpt-5-mini")
TTS_MODEL = os.getenv("OPENAI_TTS_MODEL", "gpt-4o-mini-tts")
TTS_VOICE = os.getenv("OPENAI_TTS_VOICE", "coral")

FALLBACK_TRANSCRIPT = (
    "Hi, I cannot log into my account. "
    "The password reset email never arrives, and I need access today."
)

print("Input audio exists:", INPUT_AUDIO.exists())
print("STT model:      ", STT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("Answer model:   ", ANSWER_MODEL)
print("TTS model:      ", TTS_MODEL, "/ voice:", TTS_VOICE)

## Step 3 — Load and preprocess the audio

We convert the input into **16 kHz mono** audio with FFmpeg, and fall back to librosa/soundfile if FFmpeg is missing.

If `sample_support_call.wav` is missing, the notebook continues with a prepared transcript.

> Say: "OpenAI can accept common audio formats directly, but preprocessing keeps the teaching pipeline explicit and makes the input predictable. FFmpeg is the utility layer that normalizes messy real-world audio before model calls."

In [ ]:
transcript = None
HAS_AUDIO = INPUT_AUDIO.exists()

if HAS_AUDIO:
    if shutil.which("ffmpeg"):
        cmd = [
            "ffmpeg",
            "-y",
            "-i", str(INPUT_AUDIO),
            "-ar", "16000",
            "-ac", "1",
            str(CLEAN_AUDIO),
        ]
        subprocess.run(cmd, check=True, capture_output=True)
        print("Saved", CLEAN_AUDIO, "with FFmpeg")
    else:
        audio, sample_rate = librosa.load(INPUT_AUDIO, sr=16000, mono=True)
        sf.write(CLEAN_AUDIO, audio, sample_rate)
        print("Saved", CLEAN_AUDIO, "with librosa/soundfile fallback")

    audio, sample_rate = librosa.load(CLEAN_AUDIO, sr=16000, mono=True)
    duration = len(audio) / sample_rate

    print("Sample rate:", sample_rate)
    print("Duration:", round(duration, 2), "seconds")
    print("Samples:", audio.shape)

    display(Audio(str(CLEAN_AUDIO), autoplay=False))
else:
    transcript = FALLBACK_TRANSCRIPT
    print("Missing sample_support_call.wav. Using fallback transcript:")
    print(transcript)

## Step 4 — Transcribe with OpenAI speech-to-text

Speech-to-text converts spoken language into written text. It is not reasoning; it only produces a transcript.

The cell tries the OpenAI transcription API first. If the project has no access to a transcription model, it tries a local Whisper model, and finally the prepared transcript. The demo never stops here.

> Say: "Speech-to-text is not reasoning. It converts spoken language into written text. Everything downstream depends on this transcript being good enough."

In [ ]:
stt_method = "fallback transcript"

if transcript is None:
    try:
        with CLEAN_AUDIO.open("rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model=STT_MODEL,
                file=audio_file,
                prompt=(
                    "A short customer support call about account login, "
                    "password reset emails, and urgent access."
                ),
            )
        transcript = transcription.text.strip()
        stt_method = f"OpenAI {STT_MODEL}"

    except Exception as exc:
        print("OpenAI speech-to-text unavailable:", repr(exc)[:200])
        try:
            import whisper

            asr_model = whisper.load_model("tiny")
            result = asr_model.transcribe(str(CLEAN_AUDIO), fp16=False)
            transcript = result["text"].strip()
            stt_method = "local whisper tiny"
        except Exception as local_exc:
            print("Local Whisper unavailable:", repr(local_exc)[:200])
            transcript = FALLBACK_TRANSCRIPT

print("Transcription method:", stt_method)
print("Transcript:")
print(transcript)

Expected transcript should be close to:

```text
Hi, I cannot log into my account. The password reset email never arrives, and I need access today.
```

## Step 5 — Build a small retrieval dataset

These are example transcripts from previous support calls.

> Say: "These are previous calls. In a production system, each row would link to the original audio file, timestamps, speaker metadata, ticket outcome, and permission metadata."

In [ ]:
past_calls = [
    {
        "id": "call_101",
        "topic": "billing",
        "transcript": "The customer was charged twice for the same invoice and wants a refund.",
    },
    {
        "id": "call_102",
        "topic": "password_reset",
        "transcript": "The user cannot log in because the password reset email is not arriving.",
    },
    {
        "id": "call_103",
        "topic": "delivery_delay",
        "transcript": "The customer asks why their package is delayed and wants a new delivery date.",
    },
    {
        "id": "call_104",
        "topic": "audio_issue",
        "transcript": "The microphone is not detected during video calls after a laptop update.",
    },
]

pd.DataFrame(past_calls)

## Step 6 — Generate OpenAI embeddings and retrieve similar calls

We embed the transcript, not the raw audio. For spoken business content this is usually the most robust first approach, because it captures **what was said**.

Direct audio embeddings are a different tool. They are useful for speaker similarity, music, and environmental sounds.

> Say: "We are not embedding the raw audio here. We are embedding the transcript. For spoken business content, this is usually the most robust first approach."

In [ ]:
def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


def embed_texts(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    vectors = np.array([item.embedding for item in response.data], dtype=np.float32)
    return normalize_rows(vectors)


past_texts = [item["transcript"] for item in past_calls]

past_embeddings = embed_texts(past_texts)
query_embedding = embed_texts([transcript])

scores = cosine_similarity(query_embedding, past_embeddings)[0]

results = pd.DataFrame({
    "id": [item["id"] for item in past_calls],
    "topic": [item["topic"] for item in past_calls],
    "transcript": past_texts,
    "score": scores,
}).sort_values("score", ascending=False)

results

Expected behavior: `password_reset` should be the top or near-top result.

## Step 7 — Send transcript and retrieved context to an OpenAI LLM

The LLM receives the transcript and the retrieved context. It is not listening to the original audio in this pipeline. That modular design makes the system easier to inspect and debug.

> Say: "The LLM is not listening to audio here. It receives the transcript and the retrieved context. That makes the system easier to inspect and debug."

In [ ]:
retrieved_context = results.head(2).to_dict(orient="records")

user_prompt = (
    "New call transcript:\n"
    f"{transcript}\n\n"
    "Similar past calls:\n"
    f"{retrieved_context}\n\n"
    "Task:\n"
    "1. Summarize the user issue in one sentence.\n"
    "2. Identify the likely intent.\n"
    "3. Suggest the next best support action.\n"
    "Keep the answer short and clear."
)

print(user_prompt)

In [ ]:
response = client.responses.create(
    model=ANSWER_MODEL,
    input=[
        {
            "role": "developer",
            "content": (
                "You are a concise support assistant. "
                "Use only the transcript and retrieved context. "
                "Do not invent account details, ticket IDs, or policies."
            ),
        },
        {"role": "user", "content": user_prompt},
    ],
)

answer = response.output_text.strip()
print(answer)

## Step 8 — Prepare the answer for text-to-speech

Text-to-speech works better when the text is short, clean, and free of Markdown formatting.

In [ ]:
def clean_for_tts(text: str) -> str:
    """Make LLM output easier for a speech engine to read aloud."""
    text = re.sub(r"[*_`#>-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


spoken_answer = clean_for_tts(answer)

# Keep the live voice output short enough for class.
if len(spoken_answer) > 450:
    spoken_answer = spoken_answer[:450] + "."

print(spoken_answer)

## Step 9 — Convert the generated answer back into speech

This is the text-to-speech step. The system is not understanding anything new here. It only changes the output modality from text to audio.

The cell tries OpenAI TTS first and falls back to the local `pyttsx3` engine if the project has no TTS access.

> Say: "This is text-to-speech. The system is not understanding anything new at this stage. It is only changing the output modality from text to audio."

In [ ]:
tts_method = None

try:
    with client.audio.speech.with_streaming_response.create(
        model=TTS_MODEL,
        voice=TTS_VOICE,
        input=spoken_answer,
        instructions="Speak clearly and calmly as a customer support assistant.",
        response_format="wav",
    ) as response:
        response.stream_to_file(TTS_OUTPUT)
    tts_method = f"OpenAI {TTS_MODEL} / {TTS_VOICE}"

except Exception as exc:
    print("OpenAI text-to-speech unavailable:", repr(exc)[:200])
    try:
        import pyttsx3

        engine = pyttsx3.init()
        engine.setProperty("rate", 165)
        engine.setProperty("volume", 0.9)
        engine.save_to_file(spoken_answer, str(TTS_OUTPUT))
        engine.runAndWait()

        # Some local drivers finish writing the file shortly after runAndWait().
        for _ in range(50):
            if TTS_OUTPUT.exists() and TTS_OUTPUT.stat().st_size > 0:
                break
            time.sleep(0.1)

        tts_method = "local pyttsx3"
    except Exception as local_exc:
        print("Local TTS failed too. The text answer is still available.")
        print("Error:", repr(local_exc)[:200])

if tts_method:
    print("TTS method:", tts_method)
    print("Saved", TTS_OUTPUT)

## Step 10 — Play the generated audio answer

In [ ]:
if TTS_OUTPUT.exists():
    display(Audio(str(TTS_OUTPUT), autoplay=False))
else:
    print("No", TTS_OUTPUT, "found. Use the text answer instead.")

Ask:

> "The answer now sounds confident. Does that make it more correct?"

Emphasize:

"TTS makes the output feel natural, but it does not validate it. The spoken response inherits all upstream errors from ASR, retrieval, and the LLM."

## Step 11 — The complete pipeline map

```text
sample_support_call.wav
-> sample_16k_mono.wav
-> OpenAI transcript
-> OpenAI transcript embedding
-> similar past calls
-> OpenAI LLM answer
-> OpenAI TTS audio
-> answer_openai.wav
```

> Ask: "Where could this system fail?"

Good answers:

- the microphone recording is noisy
- ASR mishears a key word, code, name, or number
- retrieval returns the wrong past call
- the LLM overgeneralizes from weak context
- the generated voice makes the answer sound more certain than it should
- the API key, rate limit, network, or account permissions fail during class

## Student mini challenge

Change the input content and rerun retrieval, LLM answer, and TTS:

```text
My package is delayed and nobody can tell me when it will arrive.
```

Question:

> "Did the top retrieved call change from password reset to delivery delay?"

Bonus:

- Change the TTS instructions to make the voice slower or more empathetic.
- Change the retrieved context from top 2 to top 1 and compare the LLM answer.
- Add one sentence that says: "Please confirm before I take action."

In [ ]:
challenge_transcript = "My package is delayed and nobody can tell me when it will arrive."

challenge_embedding = embed_texts([challenge_transcript])
challenge_scores = cosine_similarity(challenge_embedding, past_embeddings)[0]

challenge_results = pd.DataFrame({
    "id": [item["id"] for item in past_calls],
    "topic": [item["topic"] for item in past_calls],
    "score": challenge_scores,
}).sort_values("score", ascending=False)

challenge_results

## If things go wrong

| Problem | Fix |
|---|---|
| `OPENAI_API_KEY` is missing | Check `.env`, run `load_dotenv()`, and restart the notebook kernel |
| `.env` is not loaded | Keep `.env` next to this notebook, or pass the path: `load_dotenv("/path/to/.env")` |
| `No module named dotenv` | Run `pip install python-dotenv` |
| `No module named openai` | Run `pip install --upgrade openai` |
| `sample_support_call.wav` is missing | The notebook uses the fallback transcript and continues from retrieval |
| `ffmpeg` not found | Install FFmpeg: `brew install ffmpeg` or `sudo apt-get install ffmpeg` |
| `403 does not have access to model` | Set the matching `OPENAI_*_MODEL` in `.env` to a model your project can use, or let the local fallback run |
| ASR transcript is poor | Re-record the sample more clearly or use the prepared fallback transcript |
| Embedding call fails | Check API key, account access, model name, and network connection |
| LLM call fails | Set `OPENAI_ANSWER_MODEL` in `.env` to a model available in your account |
| Notebook does not play audio | Open `answer_openai.wav` from the file browser instead |
| API rate limit occurs | Fewer reruns, shorter audio, or instructor-only execution |
| Costs are a concern | Keep clips short, avoid repeated TTS generation, run the demo once from the instructor machine |

## Instructor wrap-up

| Step | What it does | What it does not do |
|---|---|---|
| Speech-to-text | Converts speech to text | Does not understand intent |
| Transcript embedding | Makes spoken content searchable | Does not reason about the issue |
| Retrieval | Finds similar prior content | Can retrieve irrelevant context |
| LLM | Summarizes and suggests action | Can hallucinate or overstate certainty |
| TTS | Speaks the answer | Does not improve correctness |

Final sentence to students:

> "A voice AI system is not one model. It is a pipeline, and every conversion step needs to be tested separately."